In [1]:
!pip install langchain
from langchain_core.documents import Document

documents = [
    Document(
        page_content="FAISS is used for efficient similarity search...",
        metadata={
            "source": "faiss_guide.pdf",
            "category": "technical",
            "date": "2024-01-10",
            "document_type": "documentation"
        }
    ),
    Document(
        page_content="Our product leverages AI for better insights...",
        metadata={
            "source": "marketing_blog.txt",
            "category": "marketing",
            "date": "2022-06-15",
            "document_type": "blog"
        }
    )
]

In [6]:
# Install the sentence-transformers library to use HuggingFaceEmbeddings
!pip install sentence-transformers

In [9]:
# Install the new recommended package for HuggingFaceEmbeddings if you haven't already
!pip install -U langchain-huggingface
# Install faiss-cpu for efficient similarity search
!pip install faiss-cpu

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings # Updated import

# Initialize HuggingFaceEmbeddings with a pre-trained model
# You can choose different models from the Hugging Face Hub, e.g., 'all-MiniLM-L6-v2'
# This model is a good balance of performance and size.
embedding_model_hf = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the FAISS vector store using the Hugging Face embedding model
# Note: This will re-embed the documents, which might take a moment.
vectorstore_hf = FAISS.from_documents(documents, embedding_model_hf)

print("FAISS vector store created successfully using Hugging Face Embeddings!")
# You can now use `vectorstore_hf` for similarity searches, for example:
# query_results_hf = vectorstore_hf.similarity_search("How does FAISS work?", k=3)
# for doc in query_results_hf:
#     print(doc.metadata)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 39.6 MB/s eta 0:00:00


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS vector store created successfully using Hugging Face Embeddings!


In [13]:
from datetime import datetime

def filtered_retrieve(query, filters=None, k=5):
    # Use the new vector store: vectorstore_hf
    results = vectorstore_hf.similarity_search(query, k=10)  # fetch more first

    if not filters:
        return results[:k]

    filtered = []

    for doc in results:
        meta = doc.metadata
        keep = True

        # Category filter
        if "category" in filters:
            if meta.get("category") != filters["category"]:
                keep = False

        # Date filter
        if "date_after" in filters:
            doc_date = datetime.strptime(meta.get("date"), "%Y-%m-%d")
            cutoff = datetime.strptime(filters["date_after"], "%Y-%m-%d")
            if doc_date < cutoff:
                keep = False

        # Document type filter
        if "document_type" in filters:
            if meta.get("document_type") != filters["document_type"]:
                keep = False

        if keep:
            filtered.append(doc)

    return filtered[:k]

In [17]:
queries = [
    "How does FAISS work?",
    "AI product benefits",
    "vector search optimization",
    "latest AI trends",
    "technical documentation for embeddings"
]

filters = {
    "category": "technical",
    "date_after": "2023-01-01"
}

for q in queries:
    print(f"\nQuery: {q}")

    # Use the new vector store: vectorstore_hf
    no_filter = vectorstore_hf.similarity_search(q, k=3)
    with_filter = filtered_retrieve(q, filters)

    print("\nWithout Filter:")
    for doc in no_filter:
        print("-", doc.metadata)

    print("\nWith Filter:")
    for doc in with_filter:
        print("-", doc.metadata)


Query: How does FAISS work?

Without Filter:
- {'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}
- {'source': 'marketing_blog.txt', 'category': 'marketing', 'date': '2022-06-15', 'document_type': 'blog'}

With Filter:
- {'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}

Query: AI product benefits

Without Filter:
- {'source': 'marketing_blog.txt', 'category': 'marketing', 'date': '2022-06-15', 'document_type': 'blog'}
- {'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}

With Filter:
- {'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}

Query: vector search optimization

Without Filter:
- {'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}
- {'source': 'marketing_blog.txt', 'category': 'ma

In [15]:
filtered_retrieve(
    "embedding techniques",
    filters={
        "category": "technical",
        "date_after": "2023-01-01"
    }
)

[Document(id='aba8c555-19d6-4d11-a667-51c9b806a4e0', metadata={'source': 'faiss_guide.pdf', 'category': 'technical', 'date': '2024-01-10', 'document_type': 'documentation'}, page_content='FAISS is used for efficient similarity search...')]

In [ ]:
filters = {"category": ["technical", "product"]}